# ClauseXplain demo
Contract in, categorised clauses + Indian-law rule flags out (HTML report).

**Setup:** attach datasets `clausexplain-indian-eval` and `scan-vs-span`, turn on **GPU (T4)** and **Internet**. Optional: add your own contract PDFs/TXTs as a dataset (any name) and they get picked up automatically.

Honest limits are printed inside each report: trained on US contracts (CUAD), 12 of 18 categories only, rule flags are candidates for lawyer review, not legal advice.

In [ ]:
import os, json, shutil, glob, subprocess, sys
BASE = "/kaggle/input/datasets/riyasuryawanshi"
EVAL_SRC, MODEL_SRC = f"{BASE}/clausexplain-indian-eval", f"{BASE}/scan-vs-span"
WORK = "/kaggle/working/demo"
CKPT = f"{MODEL_SRC}/ckpt_joint_cls_span_fold0.pt"
if os.path.exists(WORK): shutil.rmtree(WORK)
os.makedirs(WORK + "/contracts")
for f in ["pipeline.py","dataset.py","model.py","metrics.py","train_phase3.py","scan_vs_span.py","rules_engine.py",
          "make_label_sheet.py","prepare_documents.py","indian_legal_rules.csv"]:
    shutil.copy(f"{EVAL_SRC}/{f}", WORK)
import base64
# patched files (override the dataset copies)
for _n, _b in {"pipeline.py": "...", "rules_engine.py": "..."}.items():
    open(f"{WORK}/{_n}", "wb").write(base64.b64decode(_b))
# sample contracts (small Indian documents from the test set)
texts = json.load(open(f"{EVAL_SRC}/built_v2/indian_contracts.json"))
for k in ["indian_stqc_non_disclosure_agreement", "indian_mea_architect_consultancy_agreement", "indian_tender_consultancy_scc_1349"]:
    open(f"{WORK}/contracts/{k}.txt", "w", encoding="utf-8").write(texts[k])
# any extra PDFs/TXTs you uploaded as a dataset
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    if p.lower().endswith((".pdf", ".txt")) and "/clausexplain-indian-eval" not in p and "/scan-vs-span" not in p:
        shutil.copy(p, f"{WORK}/contracts/"); print("added", os.path.basename(p))
subprocess.run("pip -q install pypdf 2>&1 | tail -1", shell=True)
print(sorted(os.listdir(WORK + "/contracts")))

In [ ]:
import time
os.chdir(WORK); os.makedirs("reports", exist_ok=True)
FIND = ["Termination", "Governing Law / Jurisdiction", "Liability"]   # span head proposes + classifier verifies
for f in sorted(os.listdir("contracts")):
    t = time.time(); name = os.path.splitext(f)[0]
    r = subprocess.run([sys.executable, "pipeline.py", "--input", f"contracts/{f}", "--ckpt", CKPT,
                        "--rules", "indian_legal_rules.csv", "--out", f"reports/{name}.html", "--find"] + FIND,
                       capture_output=True, text=True)
    print(f"{f}: {time.time()-t:.0f}s |", (r.stdout.strip().splitlines() or ["(no output)"])[-1], r.stderr[-800:] if r.returncode else "")

In [ ]:
from IPython.display import HTML, display
import html as _h
first = sorted(os.listdir("reports"))
first = [f for f in first if f.endswith(".html")][0]
doc = open(f"reports/{first}", encoding="utf-8").read()
display(HTML(f'<iframe srcdoc="{_h.escape(doc)}" width="100%" height="700"></iframe>'))

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/clausexplain_demo_reports.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("reports"): z.write(f"reports/{f}", f)
print("Download clausexplain_demo_reports.zip from the Output tab; each .html opens in any browser, no setup needed.")

## Optional: live upload UI (Gradio)
Run this only if you want to demo with a fresh file live. It prints a public link (needs Internet on). Keep the HTML reports above as the safe backup.

In [ ]:
subprocess.run("pip -q install gradio 2>&1 | tail -1", shell=True)
sys.path.insert(0, WORK); os.chdir(WORK)
import gradio as gr
from pipeline import ClauseXplain, render_html, load_input
cx = ClauseXplain(CKPT, "indian_legal_rules.csv")
def run(file, cats):
    res = cx.analyze(load_input(file.name), cats or None, 0.6)
    doc = render_html(res, os.path.basename(file.name))
    return f'<iframe srcdoc="{_h.escape(doc)}" width="100%" height="800"></iframe>'
gr.Interface(run, [gr.File(label="Contract (.pdf or .txt)"),
                   gr.CheckboxGroup(["Termination","Governing Law / Jurisdiction","Liability","Warranty","Intellectual Property"], label="Find clause (optional)")],
             gr.HTML(), title="ClauseXplain").launch(share=True)